# Demo 03 — Memória

03 - MEMORIA: curto prazo, longo prazo e recuperacao just-in-time.
Aula 1 - Agentic Workflows (Especializacao em IA Generativa / UFPR).

O QUE ESTA DEMO MOSTRA, EM 4 ETAPAS:
  1. MEMORIA DE CURTO PRAZO - a janela de conversa. Finita: o que passa do
     limite e descartado, e o agente "esquece" na frente do usuario.
  2. MEMORIA DE LONGO PRAZO ESTRUTURADA - um grafo de relacoes construido a
     partir de eventos_acesso: quem compartilhou dispositivo, IP ou conta de
     destino com quem.
  3. RECUPERACAO JUST-IN-TIME - nao despejamos o grafo inteiro no prompt.
     Recuperamos so o subgrafo do cliente investigado. Isso e o que hoje se
     chama de context engineering: a janela e um orcamento, nao um deposito.
  4. O LLM redige o alerta usando SOMENTE o que foi recuperado.

O anel encontrado envolve o cliente 256 - o mesmo da reclamacao da demo 02.
As duas demos contam a mesma historia por angulos diferentes.

> **Como rodar:** abra este notebook no Google Colab e escolha *Ambiente de execução → Executar tudo*. As células já vêm na ordem certa.

> A chave da OpenRouter usada abaixo é a **chave temporária da turma**, embutida de propósito para a aula funcionar sem setup. Ela é descartável: não é uma credencial pessoal, e é revogada depois do curso.


## Ambiente (só no Colab; local com `requirements.txt` pode pular)

In [3]:
!pip install -q openai python-dotenv matplotlib networkx

## Configuração

In [4]:
import os
import sqlite3
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

CHAVE_TEMPORARIA = "REMOVIDA"   # chave da aula; o ambiente sempre vence
MODELO = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.6-luna")
CHAVE  = os.getenv("OPENROUTER_API_KEY") or CHAVE_TEMPORARIA
PASTA  = Path(globals().get("__file__", ".")).resolve().parent
DB     = next(p for p in (PASTA / "dados" / "curso_financeiro.db",
                          *(a / "dados" / "curso_financeiro.db" for a in PASTA.parents),
                          PASTA / "curso_financeiro.db") if p.exists())
FIG    = PASTA / "03-memoria-grafo.png"
JANELA = 4   # quantas mensagens cabem na janela
INVESTIGADO = 256   # cliente sob investigacao

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=CHAVE)



def chamar_llm(sistema, usuario, tentativas=2):
    """Unico helper de rede: chama o modelo sem deixar vazar traceback."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = client.chat.completions.create(
                model=MODELO,
                messages=[{"role": "system", "content": sistema},
                          {"role": "user", "content": usuario}],
            )
            return (resp.choices[0].message.content or "").strip()
        except Exception as erro:
            print(f"[AVISO] Falha na chamada ao modelo ({tentativa}/{tentativas}): "
                  f"{type(erro).__name__}: {erro}")
            if tentativa < tentativas:
                time.sleep(2)
    sys.exit("[ERRO] Nao foi possível falar com o OpenRouter. Verifique a rede e a chave.")


print(f"MEMORIA - modelo: {MODELO}")

MEMORIA - modelo: openai/gpt-5.6-luna


## ETAPA 1 - MEMORIA DE CURTO PRAZO: a janela desliza e o resto cai fora.

In [5]:
print(f"\n[1] MEMORIA DE CURTO PRAZO (janela = {JANELA} mensagens)")

conversa = [
    "cliente: minha conta foi acessada de um aparelho que não e meu",
    "agente: quando você percebeu isso?",
    "cliente: ontem a noite, recebi um aviso de login",
    "agente: você reconhece a cidade do acesso?",
    "cliente: não, era de outro estado",
    "agente: vou abrir uma apuracao de acesso indevido",
]
for i, mensagem in enumerate(conversa, 1):
    janela = conversa[max(0, i - JANELA):i]
    descartadas = i - len(janela)
    print(f"    msg {i}: janela contem {len(janela)} msg(s), {descartadas} descartada(s)")

print("    -> O primeiro turno ('aparelho que não e meu') JA SAIU da janela.")


[1] MEMORIA DE CURTO PRAZO (janela = 4 mensagens)
    msg 1: janela contem 1 msg(s), 0 descartada(s)
    msg 2: janela contem 2 msg(s), 0 descartada(s)
    msg 3: janela contem 3 msg(s), 0 descartada(s)
    msg 4: janela contem 4 msg(s), 0 descartada(s)
    msg 5: janela contem 4 msg(s), 1 descartada(s)
    msg 6: janela contem 4 msg(s), 2 descartada(s)
    -> O primeiro turno ('aparelho que não e meu') JA SAIU da janela.


## ETAPA 2 - MEMORIA DE LONGO PRAZO: um grafo que persiste entre sessoes.

Dois clientes ficam ligados quando compartilham o MESMO recurso.

In [6]:
print("\n[2] MEMORIA DE LONGO PRAZO (grafo de recursos compartilhados)")

con = sqlite3.connect(DB)
eventos = con.execute(
    "SELECT cliente_id, recurso, tipo_recurso FROM eventos_acesso"
).fetchall()
con.close()

# Agrupa clientes DISTINTOS por recurso. Contar clientes distintos (e não
# eventos) e o que evita marcar como "compartilhado" um recurso que um mesmo
# cliente acessou duas vezes.
por_recurso = {}
tipo_do_recurso = {}
for cliente_id, recurso, tipo in eventos:
    por_recurso.setdefault(recurso, set()).add(cliente_id)
    tipo_do_recurso[recurso] = tipo

compartilhados = {r: c for r, c in por_recurso.items() if len(c) > 1}
print(f"    {len(eventos)} eventos, {len(por_recurso)} recursos distintos, "
      f"{len(compartilhados)} compartilhados por mais de um cliente")

# O grafo contem SO as arestas de compartilhamento. Nao adicionamos os nos
# isolados: eles não dizem nada sobre relacao e so poluiriam o desenho.
#
# MultiGraph, e não Graph: dois clientes podem compartilhar MAIS DE UM recurso
# (ex.: 37 e 512 compartilham o dispositivo E a conta de destino do PIX). Com
# um Graph simples, a segunda ligacao sobrescreveria a primeira e perderiamos
# justamente a evidencia que reforca o anel.
G = nx.MultiGraph()
for recurso, clientes in compartilhados.items():
    lista = sorted(clientes)
    for i in range(len(lista)):
        for j in range(i + 1, len(lista)):
            G.add_edge(lista[i], lista[j], recurso=recurso,
                       tipo=tipo_do_recurso[recurso])

componentes = sorted(nx.connected_components(G), key=len, reverse=True)
print(f"    grafo: {G.number_of_nodes()} clientes conectados, "
      f"{G.number_of_edges()} ligacoes, {len(componentes)} grupo(s)")
for k, comp in enumerate(componentes, 1):
    print(f"      grupo {k}: {len(comp)} clientes -> {sorted(comp)}")


[2] MEMORIA DE LONGO PRAZO (grafo de recursos compartilhados)
    33 eventos, 24 recursos distintos, 4 compartilhados por mais de um cliente
    grafo: 8 clientes conectados, 17 ligacoes, 2 grupo(s)
      grupo 1: 5 clientes -> [37, 48, 112, 256, 512]
      grupo 2: 3 clientes -> [201, 202, 203]


## ETAPA 3 - RECUPERACAO JUST-IN-TIME

So o subgrafo do cliente investigado vai para o prompt.

In [7]:
print(f"\n[3] RECUPERACAO JUST-IN-TIME (cliente investigado: {INVESTIGADO})")

if INVESTIGADO not in G:
    sys.exit(f"    O cliente {INVESTIGADO} não tem recurso compartilhado no banco.")

anel = next(c for c in componentes if INVESTIGADO in c)
subgrafo = G.subgraph(anel)

fatos = []
for a, b, dados in sorted(subgrafo.edges(data=True),
                          key=lambda e: (e[0], e[1], e[2]["recurso"])):
    fatos.append(f"clientes {a} e {b} compartilham {dados['tipo']} '{dados['recurso']}'")

print(f"    recuperados {len(fatos)} fatos de {G.number_of_edges()} existentes "
      f"({100 * len(fatos) / G.number_of_edges():.0f}% do grafo):")
for fato in fatos:
    print(f"      - {fato}")


[3] RECUPERACAO JUST-IN-TIME (cliente investigado: 256)
    recuperados 14 fatos de 17 existentes (82% do grafo):
      - clientes 37 e 48 compartilham dispositivo 'dispositivo_X'
      - clientes 37 e 112 compartilham dispositivo 'dispositivo_X'
      - clientes 37 e 256 compartilham dispositivo 'dispositivo_X'
      - clientes 37 e 512 compartilham destino_pix 'conta_mula_9'
      - clientes 37 e 512 compartilham dispositivo 'dispositivo_X'
      - clientes 48 e 112 compartilham dispositivo 'dispositivo_X'
      - clientes 48 e 112 compartilham ip 'ip_200_10_5_9'
      - clientes 48 e 256 compartilham dispositivo 'dispositivo_X'
      - clientes 48 e 512 compartilham dispositivo 'dispositivo_X'
      - clientes 48 e 512 compartilham ip 'ip_200_10_5_9'
      - clientes 112 e 256 compartilham dispositivo 'dispositivo_X'
      - clientes 112 e 512 compartilham dispositivo 'dispositivo_X'
      - clientes 112 e 512 compartilham ip 'ip_200_10_5_9'
      - clientes 256 e 512 compartilham 

## ETAPA 4 - O LLM REDIGE O ALERTA COM BASE SO NO QUE FOI RECUPERADO

In [8]:
print("\n[4] ALERTA REDIGIDO PELO MODELO")

alerta = chamar_llm(
    "Voce e um analista de prevencao a fraude. Escreva um alerta objetivo de até "
    "4 frases para a mesa de risco. Use SOMENTE os fatos fornecidos, sem inventar "
    "nomes, valores ou datas. Termine recomendando uma acao.",
    f"Cliente sob investigacao: {INVESTIGADO}\nFatos recuperados do grafo:\n"
    + "\n".join(f"- {f}" for f in fatos),
)
print(alerta)


[4] ALERTA REDIGIDO PELO MODELO
Cliente 256 compartilha o dispositivo `dispositivo_X` com os clientes 37, 48, 112 e 512. Os clientes 48, 112 e 512 também compartilham o IP `ip_200_10_5_9`, enquanto os clientes 37 e 512 compartilham o destino PIX `conta_mula_9`. Recomenda-se manter o cliente 256 sob investigação e realizar análise aprofundada das transações e vínculos relacionados.


## GRAFICO: o anel desenhado, com o investigado destacado.

In [9]:
posicoes = nx.spring_layout(G, seed=42)
cores = ["#c0392b" if n == INVESTIGADO else
         ("#e67e22" if n in anel else "#2c5282") for n in G.nodes()]

fig, ax = plt.subplots(figsize=(7, 4.5))
nx.draw_networkx_edges(G, posicoes, ax=ax, alpha=0.45)
nx.draw_networkx_nodes(G, posicoes, ax=ax, node_color=cores, node_size=650)
nx.draw_networkx_labels(G, posicoes, ax=ax, font_size=8, font_color="white")
rotulos = {}
for a, b, d in G.edges(data=True):
    par = (a, b) if a < b else (b, a)
    rotulos[par] = (rotulos.get(par, "") + "\n" + d["recurso"]).strip()
nx.draw_networkx_edge_labels(G, posicoes, edge_labels=rotulos, ax=ax, font_size=6)
ax.set_title(f"Memoria de longo prazo: anel do cliente {INVESTIGADO} "
             f"(vermelho) e vizinhos (laranja)")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIG, dpi=110)
plt.close(fig)
print(f"\n[grafico salvo] {FIG.name}")

print("LICAO: memoria não e 'guardar tudo'. E escolher o que recuperar.")
print("A janela e um orçamento; o grafo e o arquivo. Recupere sob demanda.")


[grafico salvo] 03-memoria-grafo.png
LICAO: memoria não e 'guardar tudo'. E escolher o que recuperar.
A janela e um orçamento; o grafo e o arquivo. Recupere sob demanda.
